In [1]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt 

# ignite
from ignite.engine import Engine, create_supervised_trainer, create_supervised_evaluator, Events
from ignite.handlers import ModelCheckpoint, global_step_from_engine
from ignite.handlers import EarlyStopping

In [2]:
data = "/home/jovyan/DADOS-DIVIDIDOS"
feature_extract= True

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5, interpolation=3, fill=0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(data, x), data_transforms[x]) for x in ['train', 'val','test']}

/opt/conda/lib/python3.10/site-packages/torchvision/transforms/transforms.py:768: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  warnings.warn(


In [3]:
# Extração de features + Congelamento dos parâmetros
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False



Best hyperparameters : {'dropout1': 0.21269401273868907, 'dropout2': 0.3333736211255451, 'num_neurons_fc1': 512, 'num_neurons_fc2': 512, 'activation': 'ReLU', 'batch_size': 128, 'optimizer': 'SGD', 'lr': 0.0019134979136296384, 'momentum': 0.9899652823553813}

In [4]:
dropout_rate1 = 0.21269401273868907
dropout_rate2 = 0.3333736211255451
num_neurons_fc1 = 512
num_neurons_fc2 = 512
batch_size = 128
momentum = 0.9899652823553813
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

criterion = nn.CrossEntropyLoss()

In [5]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Subset
from sklearn.model_selection import StratifiedKFold
from ignite.metrics import Accuracy, Loss
from ignite.engine import Events, Engine, create_supervised_evaluator
from ignite.handlers import ModelCheckpoint, EarlyStopping
from ignite.contrib.handlers.param_scheduler import LRScheduler
from torch.optim.lr_scheduler import StepLR


def create_model():
    model = models.mobilenet_v3_large(pretrained=False)
    
    if feature_extract:
        for param in model.parameters():
            param.requires_grad = False
    
    set_parameter_requires_grad(model, feature_extract)

    model_mobile_net = "/home/jovyan/models/mobilenet_v3_large-model-84.pth"
    state_dict = torch.load(model_mobile_net)

    del state_dict['classifier.3.weight']
    del state_dict['classifier.3.bias']

    model.load_state_dict(state_dict, strict=False)

    num_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate1),
        nn.Linear(num_features, num_neurons_fc1),
        nn.ReLU(),
        nn.Dropout(p=dropout_rate2),
        nn.Linear(num_neurons_fc1, 2),
        nn.Softmax(dim=1)
    )
    
    return model.to(device)

def train_step(engine, batch):
    model.train()
    inputs, labels = batch[0].to(device), batch[1].to(device)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    return loss.item()

def validation_step(engine, batch):
    model.eval()
    with torch.no_grad():
        inputs, labels = batch[0].to(device), batch[1].to(device)
        outputs = model(inputs)
        return outputs, labels

n_splits = 10
train_labels = np.array([y for _, y in image_datasets['train']])
skf = StratifiedKFold(n_splits=n_splits, shuffle=True)

all_train_accs_folds = []
all_val_accs_folds = []
all_train_losses_folds = []
all_val_losses_folds = []

val_metrics = {
    "accuracy": Accuracy(),
    "loss": Loss(criterion)
}

def score_function(engine):
    return engine.state.metrics["accuracy"]

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_labels)), train_labels)):
    print(f'Fold {fold+1}/{n_splits}')
    
    # Cria um novo modelo para cada fold
    model = create_model()
    
    train_subset = Subset(image_datasets['train'], train_idx)
    val_subset = Subset(image_datasets['train'], val_idx)
    
    train_loader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # Reinicializa o otimizador e scheduler para cada fold
    params_to_update = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.SGD(params_to_update, lr=0.001, momentum=momentum)
    torch_lr_scheduler = StepLR(optimizer, step_size=10, gamma=0.1)
    scheduler = LRScheduler(torch_lr_scheduler)
    

    trainer = Engine(train_step)
    evaluator = Engine(validation_step)
    train_evaluator = create_supervised_evaluator(model, metrics=val_metrics, device=device)

    Accuracy().attach(evaluator, 'accuracy')
    Loss(criterion).attach(evaluator, 'loss')
    Accuracy().attach(train_evaluator, 'accuracy')
    Loss(criterion).attach(train_evaluator, 'loss')

    train_accs = []
    val_accs = []
    train_losses = []
    val_losses = []
    
    @trainer.on(Events.STARTED)
    def start_message():
        print(f"Start training fold {fold+1}!")
        
        with open("result_mobile_net_kfold.txt", 'a', encoding='utf-8') as file:
            file.write(f"Start training fold {fold+1}! \n\n")
            
        
    @trainer.on(Events.EPOCH_COMPLETED)
    def run_train_validation():
        train_evaluator.run(train_loader)

    @trainer.on(Events.EPOCH_COMPLETED)
    def run_validation():
        evaluator.run(val_loader)
        
    @trainer.on(Events.EPOCH_COMPLETED)
    def print_lr():
        print(f"Learning rate atual: {optimizer.param_groups[0]['lr']}")
        with open("result_mobile_net_kfold.txt", 'a', encoding='utf-8') as file:
            file.write(f"Learning rate atual: {optimizer.param_groups[0]['lr']}\n\n")
    
    @train_evaluator.on(Events.COMPLETED)
    def log_train_results():
        metrics = train_evaluator.state.metrics
        train_acc = metrics['accuracy']
        train_loss = metrics['loss']
        train_accs.append(train_acc)
        train_losses.append(train_loss)
        print(f"Fold {fold+1} - Epoch {trainer.state.epoch} - Training Accuracy: {train_acc:.3f}, Loss: {train_loss:.3f}")
        
        with open("result_mobile_net_kfold.txt", 'a', encoding='utf-8') as file:
            file.write(f"Fold {fold+1} - Epoch {trainer.state.epoch} - Training Accuracy: {train_acc:.3f}, Loss: {train_loss:.3f}\n\n")

    @evaluator.on(Events.COMPLETED)
    def log_validation_results():
        metrics = evaluator.state.metrics
        val_acc = metrics['accuracy']
        val_loss = metrics['loss']
        val_accs.append(val_acc)
        val_losses.append(val_loss)
        
        print(f"Fold {fold+1} - Epoch: {trainer.state.epoch} - Validation Accuracy: {val_acc:.3f}, Loss: {val_loss:.3f}")
        with open("result_mobile_net_kfold.txt", 'a', encoding='utf-8') as file:
            file.write(f"Fold {fold+1} - Epoch: {trainer.state.epoch} - Validation Accuracy: {val_acc:.3f}, Loss: {val_loss:.3f}\n\n")

    handler = ModelCheckpoint(
        dirname=f'models_fold_{fold+1}',
        filename_prefix='best',
        n_saved=1,
        create_dir=True,
        score_function=score_function,
        score_name="val_acc",
        require_empty=False
    )
    evaluator.add_event_handler(Events.COMPLETED, handler, {'model': model})

    es_handler = EarlyStopping(patience=20, score_function=score_function, trainer=trainer)
    evaluator.add_event_handler(Events.COMPLETED, es_handler)

    trainer.run(train_loader, max_epochs=200)


/tmp/ipykernel_322572/3729120637.py:13: DeprecationWarning: /opt/conda/lib/python3.10/site-packages/ignite/contrib/handlers/param_scheduler.py has been moved to /ignite/handlers/param_scheduler.py and will be removed in version 0.6.0.
 Please refer to the documentation for more details.
  from ignite.contrib.handlers.param_scheduler import LRScheduler


Fold 1/10


/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Start training fold 1!
Fold 1 - Epoch 1 - Training Accuracy: 0.557, Loss: 0.689
Fold 1 - Epoch: 1 - Validation Accuracy: 0.554, Loss: 0.689
Learning rate atual: 0.001
Fold 1 - Epoch 2 - Training Accuracy: 0.621, Loss: 0.680
Fold 1 - Epoch: 2 - Validation Accuracy: 0.583, Loss: 0.684
Learning rate atual: 0.001
Fold 1 - Epoch 3 - Training Accuracy: 0.622, Loss: 0.669
Fold 1 - Epoch: 3 - Validation Accuracy: 0.597, Loss: 0.673
Learning rate atual: 0.001
Fold 1 - Epoch 4 - Training Accuracy: 0.647, Loss: 0.652
Fold 1 - Epoch: 4 - Validation Accuracy: 0.647, Loss: 0.659
Learning rate atual: 0.001
Fold 1 - Epoch 5 - Training Accuracy: 0.711, Loss: 0.634
Fold 1 - Epoch: 5 - Validation Accuracy: 0.662, Loss: 0.642
Learning rate atual: 0.001
Fold 1 - Epoch 6 - Training Accuracy: 0.778, Loss: 0.611
Fold 1 - Epoch: 6 - Validation Accuracy: 0.734, Loss: 0.620
Learning rate atual: 0.001
Fold 1 - Epoch 7 - Training Accuracy: 0.839, Loss: 0.588
Fold 1 - Epoch: 7 - Validation Accuracy: 0.799, Loss: 0.

2025-02-13 07:29:36,955 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


...

Fold 8 - Epoch: 85 - Validation Accuracy: 0.920, Loss: 0.388
Learning rate atual: 0.001
Fold 9/10
Start training fold 9!
Fold 9 - Epoch 1 - Training Accuracy: 0.579, Loss: 0.691
Fold 9 - Epoch: 1 - Validation Accuracy: 0.551, Loss: 0.690
Learning rate atual: 0.001
Fold 9 - Epoch 2 - Training Accuracy: 0.669, Loss: 0.683
Fold 9 - Epoch: 2 - Validation Accuracy: 0.630, Loss: 0.684
Learning rate atual: 0.001
Fold 9 - Epoch 3 - Training Accuracy: 0.699, Loss: 0.672
Fold 9 - Epoch: 3 - Validation Accuracy: 0.703, Loss: 0.670
Learning rate atual: 0.001
Fold 9 - Epoch 4 - Training Accuracy: 0.724, Loss: 0.657
Fold 9 - Epoch: 4 - Validation Accuracy: 0.688, Loss: 0.655
Learning rate atual: 0.001
Fold 9 - Epoch 5 - Training Accuracy: 0.758, Loss: 0.641
Fold 9 - Epoch: 5 - Validation Accuracy: 0.754, Loss: 0.642
Learning rate atual: 0.001
Fold 9 - Epoch 6 - Training Accuracy: 0.792, Loss: 0.620
Fold 9 - Epoch: 6 - Validation Accuracy: 0.768, Loss: 0.623
Learning rate atual: 0.001
Fold 9 - Epoch 

2025-02-14 15:27:28,152 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Fold 9 - Epoch: 42 - Validation Accuracy: 0.877, Loss: 0.421
Learning rate atual: 0.001
Fold 10/10
Start training fold 10!
Fold 10 - Epoch 1 - Training Accuracy: 0.498, Loss: 0.695
Fold 10 - Epoch: 1 - Validation Accuracy: 0.522, Loss: 0.695
Learning rate atual: 0.001
Fold 10 - Epoch 2 - Training Accuracy: 0.554, Loss: 0.688
Fold 10 - Epoch: 2 - Validation Accuracy: 0.572, Loss: 0.689
Learning rate atual: 0.001
Fold 10 - Epoch 3 - Training Accuracy: 0.603, Loss: 0.675
Fold 10 - Epoch: 3 - Validation Accuracy: 0.565, Loss: 0.679
Learning rate atual: 0.001
Fold 10 - Epoch 4 - Training Accuracy: 0.669, Loss: 0.660
Fold 10 - Epoch: 4 - Validation Accuracy: 0.681, Loss: 0.663
Learning rate atual: 0.001
Fold 10 - Epoch 5 - Training Accuracy: 0.711, Loss: 0.644
Fold 10 - Epoch: 5 - Validation Accuracy: 0.696, Loss: 0.645
Learning rate atual: 0.001
Fold 10 - Epoch 6 - Training Accuracy: 0.784, Loss: 0.622
Fold 10 - Epoch: 6 - Validation Accuracy: 0.862, Loss: 0.618
Learning rate atual: 0.001
F

2025-02-14 20:41:49,434 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Fold 10 - Epoch: 75 - Validation Accuracy: 0.913, Loss: 0.395
Learning rate atual: 0.001
